## Reference Paper

**Strack, B., DeShazo, J.P., Gennings, C., Olmo, J.L., Ventura, S., Cios, K.J., & Clore, J.N. (2014).**
*Impact of HbA1c Measurement on Hospital Readmission Rates: Analysis of 70,000 Clinical Database Patient Records.*
BioMed Research International, 2014, Article 781670.

DOI: https://doi.org/10.1155/2014/781670

Full text: https://pmc.ncbi.nlm.nih.gov/articles/PMC3996476/

**UCI Dataset:**

Clore, J., Cios, K., DeShazo, J., & Strack, B. (2014). *Diabetes 130-US Hospitals for Years 1999-2008.* UCI Machine Learning Repository.

DOI: https://doi.org/10.24432/C5230J

---

### What the Original Paper Did

The original study used this dataset to investigate whether measuring HbA1c (glycated hemoglobin, a marker of average blood glucose over 2-3 months) during a hospital stay was associated with lower 30-day readmission rates in diabetic patients.

They filtered 74 million clinical records down to approximately 70,000 encounters meeting strict criteria: inpatient admission, diabetes diagnosis, 1-14 day stay, lab tests performed, and medications administered. They then fit a multivariable logistic regression model to test the HbA1c hypothesis while controlling for demographics, disease severity, and admission type.

**Key findings:**
- HbA1c was only measured in 18.4% of encounters despite all patients having diabetes
- When HbA1c was measured and led to a medication change, readmission rates decreased
- The effect depended on the primary diagnosis
- Strongest predictors of readmission: prior inpatient visits, discharge disposition, and admission type

---

## How Our Project Extends the Paper

The original paper used only logistic regression and focused on the HbA1c hypothesis. It did not build a general readmission prediction system, did not address class imbalance, and did not evaluate model fairness.

Our project extends this work by:
1. Treating readmission prediction as a full binary classification problem (readmitted within 30 days: yes/no)
2. Implementing and comparing four supervised models: Logistic Regression, Decision Tree, Random Forest, and XGBoost
3. Adding a K-Means clustering component to identify natural patient risk groups without using the readmission label
4. Addressing the class imbalance (9% positive class) using class-weight balancing rather than SMOTE, which is more appropriate for heavily categorical data
5. Evaluating using AUC, F1-score, Recall, and Precision which are appropriate for imbalanced clinical data, and running DeLong tests to determine whether AUC differences between models are statistically significant
6. Running Mann-Whitney U tests to confirm which features are statistically associated with readmission, Kruskal-Wallis tests to confirm racial groups have different clinical profiles, and probability distribution fitting on key features
7. Performing a formal fairness analysis using Demographic Parity Difference and Equal Opportunity Difference across racial groups, and applying post-processing threshold adjustment to mitigate the identified bias
8. Deploying the best model as an interactive web application with the fairness-adjusted thresholds applied at prediction time

The logistic regression baseline serves as a direct comparison point to the original paper's methodology.

---

## Related Work

**Liu et al. (2024)** compared 10 ML models on this dataset and found XGBoost consistently achieved the best AUC (0.65 to 0.70). Their work confirmed that no single feature dominates and that ensemble models generally outperform single models, but only by a small margin.

Source: https://jmai.amegroups.org/article/view/9179/html

---

**Cureus (2025)** compared Logistic Regression, Random Forest, XGBoost, and DNN on the same UCI dataset using an 80/20 split. XGBoost achieved AUC 0.667, LR 0.642, and RF 0.630. SHAP analysis identified prior admissions and number of medications as the top predictors.

---

**Gandra (2024)** compared six models including CATBoost and LightGBM and reported a best AUC of 0.70 using CATBoost. Studies that exceed 0.67 AUC on this dataset consistently use more advanced gradient boosting methods.

---

**Shang et al. (2021)** used Random Forest, Naive Bayes, and a Decision Tree ensemble on 100,244 records without deduplication. Random Forest had the highest AUC. Number of inpatient admissions, age, and number of emergency visits were the top predictors.

---

**Hai et al. (2022)** applied LSTM to sequential multi-encounter data and gained 0.06 AUC over the best traditional models. The gain came from access to longitudinal visit histories, not from the model architecture itself. This sets an upper bound on what single-encounter tabular models can achieve.

---

**Ding and Shah (2022)** applied logistic regression and random forest to this dataset and found Caucasian patients were over-predicted as readmission candidates relative to non-Caucasian patients. They identified the bias directionally but provided no formal fairness metrics and no mitigation.

Source: https://tony-xiayiding.github.io/2022-12-24-diabetes-readmission/

---

**arxiv Equity Study (2024)** compared DL, GBM, GLM, and Naive Bayes on this dataset and ran fairness analysis across age, gender, and race. GBM showed the lowest false discovery rate across racial groups (6 to 7%). No mitigation was applied.

---

## Notebook Structure

| Notebook | Purpose |
|---|---|
| `00_reference.ipynb` | This file. References, related work, and project framing. |
| `01_data_preprocessing.ipynb` | Data cleaning, feature engineering, target binarization, and output of `diabetes_clean.csv`. |
| `02_eda.ipynb` | Exploratory data analysis: distributions, correlations, bivariate analysis, HbA1c analysis. |
| `03_modeling.ipynb` | Four supervised models, hyperparameter tuning, evaluation, K-Means clustering, feature importance, literature benchmark. Saves all models and predictions to `models/`. |
| `04_fairness.ipynb` | Fairness analysis: baseline fairness metrics, threshold adjustment mitigation, before/after comparison, chi-squared test setup. Saves `group_thresholds.csv`. |
| `05_statistical_significance.ipynb` | DeLong tests, Mann-Whitney U tests, Kruskal-Wallis tests, probability distribution fitting, chi-squared test on fairness improvement. |
| `app.py` | Streamlit web application: EDA, model performance, statistical analysis, fairness analysis, and live prediction with fairness-adjusted thresholds. |

---

## Key Results Summary

| Metric | Value |
|---|---|
| Dataset size (after preprocessing) | 69,973 encounters |
| Positive class rate | 9.0% |
| Best model (AUC) | Soft Voting Ensemble: 0.6458 |
| Deployment model | XGBoost: AUC 0.6438, Recall 55.7% |
| LR vs RF AUC gap (DeLong p-value) | 0.0028 AUC, p = 0.517 (not significant) |
| DT vs RF AUC gap (DeLong p-value) | 0.0165 AUC, p = 0.003 (significant) |
| Mann-Whitney: all features significant | Yes, all 9 features p < 0.0001 |
| Kruskal-Wallis: racial group differences | Yes, all 5 features p < 0.001 |
| Equal Opportunity Difference (before) | 17.4 percentage points |
| Equal Opportunity Difference (after) | 1.2 percentage points |
| Chi-squared p-value after mitigation | 1.000 (no detectable racial disparity in TPR) |
| AUC cost of fairness mitigation | 0.000 (threshold adjustment, no retraining) |

---

## Full Citation List

[1] Strack, B., DeShazo, J.P., Gennings, C., Olmo, J.L., Ventura, S., Cios, K.J., & Clore, J.N. (2014). Impact of HbA1c Measurement on Hospital Readmission Rates: Analysis of 70,000 Clinical Database Patient Records. *BioMed Research International*, 2014, Article 781670. https://doi.org/10.1155/2014/781670

[2] Clore, J., Cios, K., DeShazo, J., & Strack, B. (2014). Diabetes 130-US Hospitals for Years 1999-2008. *UCI Machine Learning Repository*. https://doi.org/10.24432/C5230J

[3] Liu, et al. (2024). Comparison of machine learning models for predicting 30-day readmission rates for patients with diabetes. *Journal of Medical Artificial Intelligence*. https://jmai.amegroups.org/article/view/9179/html

[4] Mingle, D. (2017). Predicting Diabetic Readmission Rates: Moving Beyond HbA1c. *Current Trends in Biomedical Engineering and Biosciences*, 7(3).

[5] Ding, T., Shah, R. (2022). Diabetes Readmission Prediction. UC Berkeley. https://tony-xiayiding.github.io/2022-12-24-diabetes-readmission/

[6] Shang, Y., et al. (2021). Predicting Hospital Readmission in Diabetic Patients Using Random Forest and other Classification Methods. *Proceedings of 2021 IEEE Conference on Computational Intelligence in Bioinformatics and Computational Biology.*

[7] Hai, T., et al. (2022). Sequential deep learning for predicting 30-day hospital readmission from longitudinal EHR data. *Journal of Biomedical Informatics.*

[8] Gandra, N. (2024). Predicting Hospital Readmission for Diabetic Patients Using Machine Learning. *ResearchGate preprint.*

[9] DeLong, E.R., DeLong, D.M., & Clarke-Pearson, D.L. (1988). Comparing the areas under two or more correlated receiver operating characteristic curves: a nonparametric approach. *Biometrics*, 44(3), 837-845.

[10] Identifying and Mitigating Algorithmic Bias in the Safety Net. (2025). *Nature Digital Medicine.*